# IEMOCAP — Exploratory Data Analysis
## Step 1: Understanding the Dataset & Defining the Problem

### Research Question
> **Can emotional sentiment be reliably detected from speech audio alone?**

### What is IEMOCAP?
The **Interactive Emotional Motion Capture (IEMOCAP)** dataset contains recordings of actors
in spontaneous and scripted dyadic interactions. It is the most widely used benchmark for
Speech Emotion Recognition (SER) research.

Key facts:
- **5 Sessions** — each session = a different male-female actor pair
- **~10,000 utterances** total, each 1–10 seconds long
- **Multiple annotators** rated every utterance for emotion (categorical + dimensional VAD)
- **Gold standard** in SER: most published papers report results on this dataset


## 0. Setup & Data Loading

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import seaborn as sns
import librosa, librosa.display
from datasets import load_dataset
from pathlib import Path
from collections import Counter

plt.rcParams.update({"figure.dpi":130,"font.family":"sans-serif",
                     "axes.spines.top":False,"axes.spines.right":False})
FIG_DIR = Path("../reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Loading IEMOCAP from HuggingFace...")
ds = load_dataset("AbstractTTS/IEMOCAP", split="train")
df = ds.to_pandas()
print(f"Total utterances : {len(df):,}")
print(f"Columns          : {list(df.columns)}")
print(f"Audio included   : {'audio' in df.columns}")


## 1. Raw Emotion Distribution

IEMOCAP contains many emotion labels. We first examine **all labels with their counts**
to understand what we are working with before any filtering.

### Why do some labels have very few samples?
IEMOCAP was collected in scripted + spontaneous scenarios. Rare emotions
(disgust, fear) were not frequently elicited, leading to very unbalanced counts.


In [ ]:
counts = df["major_emotion"].value_counts()
total  = len(df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ecc71" if c >= 200 else "#f39c12" if c >= 50 else "#e74c3c" for c in counts.values]
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8)
for bar, cnt in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, f"{cnt:,}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("Number of utterances"); ax.set_xlabel("Emotion label")
ax.set_title("Raw Emotion Distribution in IEMOCAP", fontweight="bold", fontsize=13)
patches = [mpatches.Patch(color="#2ecc71",label=">=200 (usable)"),
           mpatches.Patch(color="#f39c12",label="50-200 (marginal)"),
           mpatches.Patch(color="#e74c3c",label="<50 (too few)")]
ax.legend(handles=patches, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_01_raw_emotion_dist.png", bbox_inches="tight"); plt.show()

print("\nDetailed counts:")
for emo, cnt in counts.items():
    pct  = cnt/total*100
    flag = "✅ keep" if cnt>=200 else "⚠️ marginal" if cnt>=50 else "❌ drop"
    print(f"  {emo:<15}: {cnt:>5} ({pct:5.1f}%)  {flag}")


## 2. Defining the 4-Class Label Set

### Standard IEMOCAP Protocol (literature convention)
Most SER papers focus on **4 classes** to ensure:
1. Enough training samples per class
2. Comparable results across papers
3. Acoustically distinct categories

| Final Label | Raw Emotions Merged | Reason for merge |
|-------------|--------------------|--------------------|
| **angry** | angry | Clear high-energy, negative valence |
| **happy** | happy + excited | Both positive valence, acoustically similar (high energy) |
| **sad** | sad | Low energy, negative valence |
| **neutral** | neutral | Mid-range in all dimensions |

### Why merge happy + excited?
Research (Busso et al., 2008) shows annotators frequently confuse happy and excited.
Both have high arousal and positive valence — merging increases happy class size significantly.


In [ ]:
LABEL_MAP = {
    "angry":     "angry",
    "happy":     "happy",
    "excited":   "happy",   # merged into happy
    "sad":       "sad",
    "neutral":   "neutral",
}
COLORS_4 = {"angry":"#e74c3c","happy":"#f39c12","sad":"#9b59b6","neutral":"#3498db"}

df["label"] = df["major_emotion"].map(LABEL_MAP)
df_4 = df.dropna(subset=["label"]).copy()

print("4-Class distribution after label mapping:")
print("=" * 55)
dist = df_4["label"].value_counts()
total_4 = len(df_4)
for lbl, cnt in dist.items():
    bar = "█" * int(cnt/dist.max()*35)
    print(f"  {lbl:<10}: {cnt:>5} ({cnt/total_4*100:.1f}%)  {bar}")
print(f"\n  Total kept : {total_4:,}")
print(f"  Dropped    : {len(df)-total_4:,} (disgust, fear, surprise, other)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
wedge_colors = [COLORS_4[l] for l in dist.index]
axes[0].pie(dist.values, labels=dist.index, colors=wedge_colors, autopct="%1.1f%%",
            startangle=140, pctdistance=0.75,
            wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[0].set_title("4-Class Distribution (pie)", fontweight="bold")
bars2 = axes[1].bar(dist.index, dist.values, color=wedge_colors, edgecolor="white")
for bar, cnt in zip(bars2, dist.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+15, f"{cnt:,}",
                 ha="center", fontsize=10, fontweight="bold")
axes[1].set_ylabel("Count"); axes[1].set_title("4-Class Distribution (bar)", fontweight="bold")
plt.suptitle("Label Distribution After Merging Happy+Excited", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_02_4class_dist.png", bbox_inches="tight"); plt.show()

print("\n📌 Class imbalance present → use UAR (Unweighted Average Recall) not Accuracy!")


## 3. Session Structure & LOSO Cross-Validation

### Why LOSO?
Each session in IEMOCAP contains a **unique pair of actors**.
If we split randomly (e.g. 80/20), the model may learn the *speaker's voice* rather than the emotion.

**LOSO (Leave-One-Session-Out)**: In each fold, train on 4 sessions, test on the held-out session.
This guarantees the test speaker was **never seen** during training → truly speaker-independent evaluation.


In [ ]:
# Extract session ID from the 'file' field
df_4["session"] = df_4["file"].str.extract(r"(Ses0\d)", expand=False)
df_4["session"] = df_4["session"].fillna(df_4["file"].str[:5])

session_dist = df_4.groupby(["session","label"]).size().unstack(fill_value=0)
print("Utterances per session and emotion:")
print(session_dist.to_string())
print(f"\nTotal sessions: {df_4['session'].nunique()}")

fig, ax = plt.subplots(figsize=(11, 5))
session_dist.plot(kind="bar", ax=ax, color=[COLORS_4[c] for c in session_dist.columns],
                  edgecolor="white", linewidth=0.5)
ax.set_xlabel("Session"); ax.set_ylabel("Number of utterances")
ax.set_title("Utterances per Session × Emotion\n(used for LOSO cross-validation)", fontweight="bold")
ax.legend(title="Emotion", bbox_to_anchor=(1.01,1), fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_03_session_dist.png", bbox_inches="tight"); plt.show()

print("\n📌 LOSO folds:")
for sess in sorted(df_4["session"].unique()):
    n_test  = (df_4["session"]==sess).sum()
    n_train = total_4 - n_test
    print(f"  Fold (test={sess}): train={n_train:,}  test={n_test:,}")


## 4. Gender Distribution

IEMOCAP actors are always in male-female pairs.
Understanding gender distribution matters because:
- Pitch (F0) differs significantly between male and female voices
- A biased gender split could inflate or deflate results


In [ ]:
gender_label = df_4.groupby(["gender","label"]).size().unstack(fill_value=0)
print("Gender × Emotion distribution:"); print(gender_label.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (sess_col, title) in zip(axes,[("gender","By Gender"),("session","By Session")]):
    grp = df_4.groupby([sess_col,"label"]).size().unstack(fill_value=0)
    grp.plot(kind="bar", ax=ax, color=[COLORS_4[c] for c in grp.columns], edgecolor="white")
    ax.set_title(title, fontweight="bold"); ax.set_ylabel("Count"); ax.set_xlabel("")
    ax.tick_params(axis="x",rotation=0); ax.legend(title="Emotion", fontsize=8)
plt.suptitle("Distribution by Gender and Session", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_04_gender_session.png", bbox_inches="tight"); plt.show()


## 5. Signal Length Analysis

### Why does signal length matter?
Neural networks expect fixed-size inputs. We need to decide:
- **Maximum clip length** (cut longer clips)
- **Minimum clip length** (discard or pad shorter clips)
- **Standard window** (pad/trim all clips to this length)

We compute duration using the audio sample count and sampling rate.


In [ ]:
print("Computing clip durations...")
durations = []
for i, item in enumerate(ds):
    emo = item["major_emotion"]
    if LABEL_MAP.get(emo) is None:
        continue
    y  = np.array(item["audio"]["array"])
    sr = item["audio"]["sampling_rate"]
    durations.append({"duration":len(y)/sr, "label":LABEL_MAP[emo], "emotion":emo})
    if (i+1) % 1000 == 0:
        print(f"  {i+1:,} / {len(ds):,}")

dur_df = pd.DataFrame(durations)
print(f"\nDuration statistics (seconds):")
print(dur_df["duration"].describe().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(dur_df["duration"], bins=60, color="#2980b9", edgecolor="white", alpha=0.85)
for pct, color, lbl in [(95,"#e74c3c","95th pct"),(75,"#f39c12","75th pct"),(50,"#2ecc71","median")]:
    val = dur_df["duration"].quantile(pct/100)
    axes[0].axvline(val, color=color, linestyle="--", linewidth=1.5, label=f"{lbl}={val:.1f}s")
axes[0].set_xlabel("Duration (seconds)"); axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Clip Durations", fontweight="bold")
axes[0].legend(fontsize=9)

dur_df.boxplot(column="duration", by="label", ax=axes[1],
               boxprops=dict(color="#2980b9"), medianprops=dict(color="#e74c3c",linewidth=2))
axes[1].set_title("Duration by Emotion Class", fontweight="bold")
axes[1].set_xlabel("Emotion"); axes[1].set_ylabel("Duration (seconds)")
plt.suptitle("", fontsize=1)
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_05_durations.png", bbox_inches="tight"); plt.show()

p95 = dur_df["duration"].quantile(0.95)
print(f"\n📌 Recommendation:")
print(f"   Use {p95:.1f}s as maximum clip length (95th percentile).")
print(f"   Clips longer → trim to {p95:.1f}s. Clips shorter → zero-pad.")
print(f"   This keeps {(dur_df['duration']<=p95).mean()*100:.1f}% of clips intact.")


## 6. Audio Visualisation per Emotion

Visualising the waveform and **Mel-Spectrogram** for a representative utterance of each class.

### What is a Mel-Spectrogram?
- X-axis: time | Y-axis: frequency (Mel scale, matches human hearing) | Color: energy (dB)
- Angry speech: **wide, bright, high-frequency** energy
- Sad speech: **narrow, dark, concentrated** in low frequencies
- This 2D image is directly usable as input to a **CNN**


In [ ]:
TARGET = ["angry","happy","sad","neutral"]
found  = {}
for item in ds:
    emo = item["major_emotion"]
    mapped = LABEL_MAP.get(emo)
    if mapped in TARGET and mapped not in found:
        found[mapped] = item
    if len(found) == len(TARGET): break

fig, axes = plt.subplots(len(TARGET), 2, figsize=(16, 11))
for row, lbl in enumerate(TARGET):
    item = found[lbl]
    y_   = np.array(item["audio"]["array"], dtype=np.float32)
    sr_  = item["audio"]["sampling_rate"]
    color= COLORS_4[lbl]
    # Waveform
    librosa.display.waveshow(y_, sr=sr_, ax=axes[row,0], color=color, alpha=0.8)
    axes[row,0].set_title(f"{lbl.upper()} — Waveform  ({len(y_)/sr_:.2f}s)", fontweight="bold", color=color)
    axes[row,0].set_ylabel("Amplitude")
    # Mel-Spectrogram
    S    = librosa.feature.melspectrogram(y=y_, sr=sr_, n_mels=128, fmax=8000)
    S_db = librosa.power_to_db(S, ref=np.max)
    img  = librosa.display.specshow(S_db, sr=sr_, hop_length=512,
                                    x_axis="time", y_axis="mel",
                                    ax=axes[row,1], cmap="magma")
    axes[row,1].set_title(f"{lbl.upper()} — Mel-Spectrogram", fontweight="bold", color=color)
    fig.colorbar(img, ax=axes[row,1], format="%+2.0f dB")

plt.suptitle("Waveform & Mel-Spectrogram per Emotion Class\n(Used as CNN input in deep learning approach)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_06_spectrograms.png", bbox_inches="tight"); plt.show()
print("📌 Visual differences are clear → CNN can exploit these patterns")


## 7. Annotator Agreement Analysis

### The 'Gold Tip' from the plan
IEMOCAP stores **soft labels** — each utterance was rated by multiple annotators.
Training on utterances with **low agreement** (annotators disagreed) adds noise.

The soft label columns (angry, happy, sad, etc.) contain the fraction of annotators who chose that label.
Agreement = max soft label value. High agreement → reliable ground truth.


In [ ]:
soft_cols = ["angry","happy","excited","sad","neutral","frustrated","surprise","fear","disgust"]
soft_cols = [c for c in soft_cols if c in df_4.columns]

df_4["agreement"] = df_4[soft_cols].max(axis=1)
df_4["n_annotators_agree"] = (df_4[soft_cols] == df_4[soft_cols].max(axis=1).values.reshape(-1,1)).sum(axis=1)

print("Annotator agreement statistics:")
print(df_4["agreement"].describe().round(3).to_string())

# Threshold analysis
for thresh in [0.5, 0.6, 0.67, 0.75, 1.0]:
    kept = (df_4["agreement"] >= thresh).sum()
    print(f"  Threshold >= {thresh:.2f}: keep {kept:>5} / {len(df_4):,} ({kept/len(df_4)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df_4["agreement"], bins=30, color="#2980b9", edgecolor="white", alpha=0.85)
axes[0].axvline(0.67, color="#e74c3c", linestyle="--", linewidth=2, label="Threshold 0.67 (2/3 agree)")
axes[0].set_xlabel("Agreement score (max soft label)"); axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Annotator Agreement", fontweight="bold")
axes[0].legend()
df_4.groupby("label")["agreement"].mean().plot(kind="bar", ax=axes[1],
    color=[COLORS_4[l] for l in df_4.groupby("label")["agreement"].mean().index],
    edgecolor="white")
axes[1].set_title("Mean Agreement by Emotion Class", fontweight="bold")
axes[1].set_xlabel("Emotion"); axes[1].set_ylabel("Mean Agreement"); axes[1].tick_params(axis="x",rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_07_agreement.png", bbox_inches="tight"); plt.show()

# High-agreement subset
df_hq = df_4[df_4["agreement"] >= 0.67].copy()
print(f"\n📌 High-quality subset (agreement >= 0.67): {len(df_hq):,} utterances")
print("   Class distribution in high-quality subset:")
print(df_hq["label"].value_counts().to_string())


## 8. VAD Dimensional Analysis

IEMOCAP also provides dimensional ratings: **Valence, Arousal, Dominance**.

| Dimension | Low | High |
|-----------|-----|------|
| **Valence (EmoVal)** | Negative (sad, angry) | Positive (happy, excited) |
| **Arousal (EmoAct)** | Calm (sad, neutral) | Active (angry, excited) |
| **Dominance (EmoDom)** | Submissive (sad, fear) | Dominant (angry) |

These dimensions are useful for understanding **why** certain emotions are confused by the model.


In [ ]:
vad_stats = df_4.groupby("label")[["EmoVal","EmoAct","EmoDom"]].agg(["mean","std"]).round(2)
print("VAD statistics per class:"); print(vad_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dim in zip(axes, ["EmoVal","EmoAct","EmoDom"]):
    for lbl, color in COLORS_4.items():
        data = df_4[df_4["label"]==lbl][dim].dropna()
        data.plot(kind="kde", ax=ax, color=color, linewidth=2.2, label=lbl)
    ax.set_title(dim, fontweight="bold"); ax.set_xlabel("Score (1-5)"); ax.legend(fontsize=9)
    ax.axvline(3.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
plt.suptitle("VAD Dimensional Distributions by Emotion Class\n(Separated curves = dimension discriminates emotions)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"EDA_08_vad.png", bbox_inches="tight"); plt.show()

print("\n📌 Key observations:")
print("  Valence: happy >> neutral > sad ≈ angry (angry is high-arousal but low-valence)")
print("  Arousal: angry ≈ happy >> neutral > sad")
print("  This explains common model confusion: angry vs happy (same arousal, different valence)")


## 9. EDA Summary & Roadmap

### Key Findings

| Finding | Implication |
|---------|-------------|
| 4 classes after merging happy+excited | Standard protocol, comparable to literature |
| Class imbalance (neutral > happy) | Use **UAR** not accuracy as metric |
| 5 sessions with different speakers | Use **LOSO** cross-validation |
| Signal duration: median ~3s, 95th pct ~7s | Trim/pad clips to **4–5 seconds** |
| Angry vs happy confusion expected | Both have high arousal (EmoAct) |
| Annotator agreement: 67% threshold recommended | Filter to ~70% of data for clean training |

### Next Steps (from the roadmap)

**Week 1 — Done ✅**: Dataset loaded, 4 classes defined, LOSO structure confirmed

**Week 2 — Next**: Feature extraction
- **Approach A**: MFCCs (40 coefficients) + Mel-Spectrograms via `librosa`
- **Approach B**: Raw audio → Wav2Vec 2.0 embeddings (higher performance)

**Week 3**: Train baseline CNN on spectrograms or Random Forest on MFCCs

**Week 4**: Fine-tune Wav2Vec 2.0 (expected: UAR ~65-75%)

**Week 5**: Error analysis + answer research question
